# HerBERT fine-tuning na PolEmo2

Colab: Runtime → **T4 GPU**; opcjonalnie `HF_TOKEN` w Secrets.

| Tryb | Zadanie | Wyniki |
|---|---|---|
| LoRA (PEFT) | tekst | `data/HerbertForSequenceClassification/lora/` |
| head | tekst | `data/HerbertForSequenceClassification/head/` |
| LoRA (PEFT) | tokeny | `data/HerbertForTokenClassification/lora/` |
| head | tokeny | `data/HerbertForTokenClassification/head/` |

In [ ]:
REPO = "https://github.com/mtorzynska/nlp_language_models_lab.git"
BRANCH = "feature/herbert-finetuning"

!git clone -b $BRANCH $REPO
%cd nlp_language_models_lab

In [ ]:
# torchao: preinstalowana wersja zbyt stara dla peft
!pip install -q -U datasets transformers peft evaluate scikit-learn loguru matplotlib pyarrow torchao

In [ ]:
# Opcjonalnie: token HF z sekretów Colab (unika throttlingu przy pobieraniu modelu)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN nie ustawiony — kontynuuję anonimowo")

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'BRAK — zmień runtime na T4'}")

def pokaz_wyniki(run):
    import json
    from pathlib import Path
    from IPython.display import Image, display

    run = Path(run)
    if not (run / "metrics.json").exists():
        print(f"Brak wyników w {run} — uruchom najpierw komórkę treningową.")
        return
    print(json.dumps(json.loads((run / "metrics.json").read_text()), indent=2))
    if (run / "confusion_matrix.png").exists():
        display(Image(filename=str(run / "confusion_matrix.png")))

In [ ]:
# Przygotowanie danych (parquet są w .gitignore, generujemy na miejscu)
!python -m src.data.prepare

## 1. Tekst — LoRA (PEFT)

In [ ]:
# ~15-20 min na T4. Na próbę: dodaj --max-train 1000 --epochs 1
!python -m src.models.train --task text --mode lora

### Wyniki — LoRA (tekst)

In [ ]:
pokaz_wyniki("data/HerbertForSequenceClassification/lora")

## 2. Tekst — head (baseline)

In [ ]:
!python -m src.models.train --task text --mode head

### Wyniki — head (tekst)

In [ ]:
pokaz_wyniki("data/HerbertForSequenceClassification/head")

## 3. Tokeny — LoRA (PEFT)

In [ ]:
!python -m src.models.train --task token --mode lora

### Wyniki — LoRA (tokeny)

In [ ]:
pokaz_wyniki("data/HerbertForTokenClassification/lora")

## 4. Tokeny — head (baseline)

In [ ]:
!python -m src.models.train --task token --mode head

### Wyniki — head (tokeny)

In [ ]:
pokaz_wyniki("data/HerbertForTokenClassification/head")

## 5. Podsumowanie

In [ ]:
import json
from pathlib import Path

runs = [
    ("text", "lora"),
    ("text", "head"),
    ("token", "lora"),
    ("token", "head"),
]

print(f"{'task':<6} {'mode':<5} {'f1 (macro)':>10} {'accuracy':>9}")
for task, mode in runs:
    path = Path(f"data/HerbertFor{'Sequence' if task == 'text' else 'Token'}Classification/{mode}/metrics.json")
    if path.exists():
        m = json.loads(path.read_text())
        print(f"{task:<6} {mode:<5} {m['f1_score']:>10.4f} {m['accuracy']:>9.4f}")
    else:
        print(f"{task:<6} {mode:<5} {'—':>10} {'—':>9}")

## Drive (opcjonalnie)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/lab2_results
!cp -r data/HerbertForSequenceClassification /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak wyników (tekst)'
!cp -r data/HerbertForTokenClassification /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak wyników (tokeny)'
!cp -r models /content/drive/MyDrive/lab2_results/ 2>/dev/null || echo 'brak zapisanego modelu'